# Survival models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pedroliman/heormodel/blob/main/docs/_notebooks/survival-models.ipynb)

In [ ]:
# Install heormodel from PyPI.
%pip install -q heormodel[survival]

This tutorial shows how a fitted parametric survival curve becomes an input both engines already accept, and derives the mathematics that makes the two representations agree: sampled event times for the individual-level `MicrosimModel.continuous` and per-cycle death probabilities for the cohort `MarkovModel`. It writes out each step at the level a graduate student would want, from the survival function through the censored likelihood to the sampling of fitted uncertainty. It builds on the [microsimulation engine tutorial](https://pedroliman.github.io/heormodel/tutorials/microsim.html) and the [Markov cohort tutorial](https://pedroliman.github.io/heormodel/tutorials/mdm-cohort.html), and carries the fitted uncertainty through [`run_psa`](https://pedroliman.github.io/heormodel/tutorials/voi.html) like any other parameter draw. The full script is [`examples/survival_models.py`](https://github.com/pedroliman/heormodel/blob/main/examples/survival_models.py). Install the survival extra with `uv pip install 'heormodel[survival]'`.

## Defining the survival curve

Write the survival time as the non-negative random variable $T$ with density $f$. Four functions describe its distribution and each determines the others. The survival function $S(t) = \Pr(T > t)$ is the probability of outliving $t$; the hazard $h(t) = f(t) / S(t)$ is the event rate among those still alive at $t$; the cumulative hazard $H(t) = \int_0^t h(u)\,du$ accumulates that rate; and the density itself is $f(t) = h(t) S(t) = -S'(t)$. Starting from $h = f/S = -S'/S = -\tfrac{d}{dt}\log S$ and integrating gives the chain that ties them together,

$$
H(t) = \int_0^t h(u)\,du = -\log S(t), \qquad S(t) = e^{-H(t)}, \qquad f(t) = h(t)\,e^{-H(t)}.
$$

`heormodel.survival` stores the cumulative hazard rather than the survival function for two reasons that matter later. It is the additive quantity under competing risks, so independent causes combine by summing their cumulative hazards, and it stays numerically well behaved where $S(t)$ has underflowed to zero, so the event-time sampler can invert it far into the tail.

The reference curve is Weibull in the accelerated-failure-time parameterization, with shape $k$ and scale $\lambda$ in years,

$$
S(t) = \exp\!\left[-\left(\tfrac{t}{\lambda}\right)^{k}\right], \qquad
h(t) = \frac{k}{\lambda}\left(\frac{t}{\lambda}\right)^{k-1}, \qquad
H(t) = \left(\frac{t}{\lambda}\right)^{k}.
$$

The shape sets the direction of the hazard: $k > 1$ makes it rise with time, the pattern of a progressive disease; $k < 1$ makes it fall; and $k = 1$ collapses to the constant-hazard exponential. The example uses $k = 1.2$ and $\lambda = 6.0$, so survival at $t = \lambda$ is $e^{-1}$ regardless of shape, and the rising hazard passes through $k/\lambda$ there.

In [ ]:
from heormodel import survival as sv

reference = sv.weibull(shape=1.2, scale=6.0)     # the reference curve
round(float(reference.survival(6.0)), 4), round(float(reference.hazard_at(6.0)), 4)

Survival at the scale is 0.3679, which is $e^{-1}$ to four places, and the hazard there is 0.2, which is $k/\lambda = 1.2/6.0$. Both confirm the analytic curve is the one the code holds.

## Computing discounted life expectancy

The single number both engines target is the discounted life expectancy, and it has a closed form against which every simulated estimate can be checked. Life-years accrue at rate one per year survived, so undiscounted life expectancy is the area under the survival curve. Integration by parts on $\mathbb{E}[T] = \int_0^\infty t\,f(t)\,dt$ with $f = -S'$ moves the integral onto the survival function, since the boundary term $-t\,S(t)$ vanishes at both ends for any distribution with a finite mean,

$$
\mathbb{E}[T] = \int_0^\infty S(t)\,dt = \lambda\,\Gamma\!\left(1 + \tfrac{1}{k}\right),
$$

where the Weibull value follows from the substitution $u = (t/\lambda)^k$.

Discounting at a continuous annual rate $r$ replaces the constant accrual rate of one with the weight $e^{-rt}$ on each future year, giving the target,

$$
\text{discounted life expectancy} = \int_0^\infty e^{-rt} S(t)\,dt = \frac{1 - \mathbb{E}\!\left[e^{-rT}\right]}{r}.
$$

The second equality is the same integration by parts carried through with the discount weight, and it reads the discounted expectancy as the mean discount factor applied over the lifetime: the quantity $\mathbb{E}[e^{-rT}]$ is the Laplace transform of the density at $r$. The Weibull Laplace transform has no elementary form, so we evaluate the integral numerically at a 3% rate.

In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.special import gamma

DISCOUNT_RATE = 0.03
undiscounted = 6.0 * gamma(1 + 1 / 1.2)
discounted_le = quad(lambda t: np.exp(-DISCOUNT_RATE * t) * reference.survival(t), 0, np.inf)[0]
round(undiscounted, 5), round(discounted_le, 5)

Undiscounted life expectancy is 5.64394 years and discounting to 3% reduces it to 4.92709 years. The second number is the target: every engine below should reproduce 4.92709 up to its own approximation error, and any larger gap signals a coding difference rather than Monte Carlo noise.

## Sampling event times for an individual model

`MicrosimModel.continuous` gives each person a death time and lets the earliest competing event decide their fate, so it needs a sampler that draws $T$ from an arbitrary curve. The tool is the probability integral transform applied to the cumulative hazard: if $T$ has cumulative hazard $H$, then $H(T)$ is a unit-exponential random variable, because

$$
\Pr\!\big(H(T) > e\big) = \Pr\!\big(T > H^{-1}(e)\big) = S\!\big(H^{-1}(e)\big) = e^{-e}.
$$

Reading the identity backwards gives the sampler: draw a threshold $E \sim \text{Exponential}(1)$ and return the time $t$ solving $H(t) = E$. For the Weibull the solve is closed-form, inverting $H(t) = (t/\lambda)^k = E$ to $t = \lambda\,E^{1/k}$; for a curve with no analytic inverse, `sample_time` inverts $H$ by bisection, which needs nothing but the cumulative-hazard function and so works for mixtures, spliced tails, and adapted fits alike.

The engine reads the sampler through a competing-risks latent-time construction. Each destination state $j$ reachable from the current state carries its own cause-specific curve, the model draws a latent time $T_j$ from each, and the person moves to the destination with the earliest time, $T = \min_j T_j$, at that time. Independent latent times make cause-specific cumulative hazards add, $H_{\text{total}}(t) = \sum_j H_j(t)$, so the realized survival curve is the product of the cause-specific survival curves and the chance of exiting by cause $j$ is that cause's share of the total hazard. The alive-and-dead model has one cause, so the column below fills only the death time and leaves the alive-to-alive time infinite, since staying alive is not a timed event.

In [ ]:
import pandas as pd
from heormodel.models import MicrosimModel
from heormodel.run import run_psa

STATES, ARM = ("alive", "dead"), "Standard care"

def event_times(params, intervention, state, attrs, rng):
    curve = sv.weibull(params["shape"], params["scale"])  # curve for this draw
    times = np.full((len(state), 2), np.inf)              # columns: to alive, to dead
    alive = state == 0
    times[alive, 1] = curve.sample_time(rng, int(alive.sum()))
    return times

def reward_rates(params, intervention, state, attrs):
    alive = (state == 0).astype(float)
    return np.zeros(len(state)), alive  # no cost; one life-year per year alive

continuous = MicrosimModel.continuous(
    states=STATES, event_times=event_times, state_reward_rates=reward_rates,
    interventions=[ARM], horizon=80.0, n_individuals=200_000,
    discount_rate=DISCOUNT_RATE, effect="lifeyears")

Running the model on the true parameters should return the analytic 4.92709 up to Monte Carlo error, which a cohort of 200,000 people keeps small.

In [ ]:
draws = pd.DataFrame({"shape": [1.2], "scale": [6.0]}, index=pd.RangeIndex(1, name="iteration"))
run_psa(continuous, draws, seed=1, sequential=True).outcomes.summary().loc[ARM, "lifeyears"]

The sampled discounted life expectancy is 4.91979 years against the analytic 4.92709, a difference of 0.007 years. The gap is the sampling error of 200,000 death times, not a modeling discrepancy, so the inverse-transform sampler reproduces the curve.

## Building per-cycle transition probabilities for a cohort model

A cohort model advances a state vector on a fixed cycle grid, so instead of event times it needs the probability of dying during each cycle. Conditioning on survival to the start of cycle $k$, that probability is

$$
q_k = \Pr\!\big(k h < T \le (k+1)h \mid T > kh\big) = 1 - \frac{S\big((k+1)h\big)}{S(kh)},
$$

for cycle length $h$. The conditional survival factor $S((k+1)h) / S(kh) = e^{-\Delta H_k}$, where $\Delta H_k = H((k+1)h) - H(kh)$ is the cumulative-hazard increment over the cycle, which is the quantity `to_transition_matrix` computes for every cause.

For competing causes the per-cycle probabilities cannot be read off one at a time, because the causes share the cycle. `to_transition_matrix` treats the increments as a piecewise-constant transition intensity $Q_k$ over the cycle, with off-diagonal entries the cause increments and each diagonal entry the negative row sum. The transition-probability matrix solves the Kolmogorov forward equation $P'(t) = P(t)\,Q_k$ with $P(0) = I$, whose solution over a unit cycle is the matrix exponential $P = e^{Q_k}$; its rows sum to one by construction and it holds the relative cause mix fixed within the cycle. For the single decrement here the intensity is $Q_k = \left(\begin{smallmatrix} -\Delta H_k & \Delta H_k \\ 0 & 0 \end{smallmatrix}\right)$, and its exponential has off-diagonal entry $1 - e^{-\Delta H_k}$, so the matrix construction returns exactly the $q_k$ above.

In [ ]:
from heormodel.models import CohortSpec, MarkovModel

N_CYCLES = 60

def transitions_and_rewards(params, intervention):
    curve = sv.weibull(params["shape"], params["scale"])
    transition = sv.to_transition_matrix({("alive", "dead"): curve}, STATES, N_CYCLES)
    return CohortSpec(transition, np.zeros(2), np.array([1.0, 0.0]))

cohort = MarkovModel(
    states=STATES, interventions=[ARM], transitions_and_rewards=transitions_and_rewards,
    n_cycles=N_CYCLES, initial_state="alive", discount_rate=DISCOUNT_RATE,
    cycle_correction="half_cycle", effect="lifeyears")
cohort.evaluate(draws).summary().loc[ARM, "lifeyears"]

The cohort returns 4.94531 years, 0.018 above the analytic 4.92709. The residual is the half-cycle correction's discretization of a curve whose hazard changes within the year, not an error in the transition probabilities, so a model author obtains the same discounted life expectancy whether the same curve drives the individual or the cohort engine.

## Fitting the curve to censored data

A real analysis estimates the curve from a trial in which some patients are still alive at the data cut, so the likelihood must credit a censored patient for surviving without recording a death time. Let patient $i$ contribute an observed time $y_i = \min(T_i, c_i)$, the smaller of the true event time and the censoring time, and an event indicator $d_i$, one for a death and zero for a censored observation. A death of known time contributes its density $f(y_i)$; a censored observation contributes only that the patient outlived $y_i$, the probability $S(y_i)$. Under censoring that is independent of the event time, the two combine into one factor per patient and the sample likelihood is their product,

$$
L(k, \lambda) = \prod_i f(y_i)^{d_i}\, S(y_i)^{1 - d_i}, \qquad
\ell(k, \lambda) = \sum_i \Big[ d_i \log h(y_i) - H(y_i) \Big],
$$

where the log-likelihood uses $f = h\,e^{-H}$ to split each factor into $d_i \log h(y_i)$ and $-H(y_i)$.

Administrative censoring at a fixed calendar cut is independent of the event time, so the assumption holds by design here. The Weibull score equations have no closed-form root, so the fit maximizes $\ell$ numerically. We generate a 300-patient sample from the true curve, censor at 12 years, and fit.

In [ ]:
from lifelines import WeibullFitter

rng = np.random.default_rng(20260714)
event_time = reference.sample_time(rng, 300)    # a 300-patient trial
observed = np.minimum(event_time, 12.0)          # administrative censoring at 12 years
fit = WeibullFitter().fit(observed, (event_time <= 12.0).astype(float))
round(fit.rho_, 3), round(fit.lambda_, 3)        # fitted shape and scale

The fit recovers shape 1.203 and scale 5.906 against the true 1.2 and 6.0. The estimates sit within a percent of truth from 300 patients, close enough that the fitted curve tracks the reference over the follow-up window.

## Carrying fitted uncertainty into a probabilistic analysis

The point estimate hides how much the finite sample leaves undetermined, and a cost-effectiveness analysis needs that uncertainty propagated. Maximum-likelihood theory supplies it: the estimator $\hat\theta = (\hat k, \hat\lambda)$ is asymptotically normal, $\hat\theta \approx \mathcal{N}\big(\theta, \mathcal{I}(\theta)^{-1}\big)$, with covariance the inverse Fisher information $\mathcal{I}(\theta) = -\mathbb{E}[\partial^2 \ell / \partial\theta\,\partial\theta^\top]$. In practice the fit reports the observed information, the negative Hessian of the log-likelihood at $\hat\theta$, and its inverse $\hat V$ estimates the covariance.

`sample_params` draws parameter vectors from $\mathcal{N}(\hat\theta, \hat V)$ and places them on the `iteration` index so they mix with every other parameter draw; `from_lifelines` rebuilds the curve at each draw. This is a normal approximation to the sampling distribution, adequate at 300 patients but not exact, and it is taken on the natural parameter scale, so a wide draw could in principle stray to a non-positive shape or scale. `run_psa` then turns the parameter distribution into a distribution of life expectancies with no special case.

In [ ]:
def fitted_event_times(params, intervention, state, attrs, rng):
    curve = sv.from_lifelines(fit, params)  # curve at this draw's sampled parameters
    times = np.full((len(state), 2), np.inf)
    alive = state == 0
    times[alive, 1] = curve.sample_time(rng, int(alive.sum()))
    return times

fitted = MicrosimModel.continuous(
    states=STATES, event_times=fitted_event_times, state_reward_rates=reward_rates,
    interventions=[ARM], horizon=80.0, n_individuals=20_000, discount_rate=DISCOUNT_RATE,
    effect="lifeyears")
params = sv.sample_params(fit, 1_000, seed=1)
life_years = run_psa(fitted, params, seed=2).outcomes.effects_wide()[ARM]
round(life_years.mean(), 3), np.round(np.percentile(life_years, [2.5, 97.5]), 3)

The probabilistic discounted life expectancy is 4.865 years with a 95% credible interval of 4.432 to 5.264, which contains the analytic 4.927. The interval is the estimation uncertainty from 300 patients expressed on the decision scale; it narrows toward the true value as the trial grows, and it flows unchanged into the cost-effectiveness and value-of-information analyses covered in the [value of information tutorial](https://pedroliman.github.io/heormodel/tutorials/voi.html).